Adapted from code for data extraction for work on overabundance in Latin

In [1]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import re
import math

# Working on LASLA tokens

In [2]:
lasla_tokens_UDset = pd.read_csv("query_lasla_tokens_UDset.csv")

In [3]:
def remove_non_numeric(lemma):
    lemma = float(re.sub("[^0-9]","",lemma))
    return(lemma)

In [4]:
lasla_tokens_UDset["lila_id_lemma_float"] = lasla_tokens_UDset["lila_id_lemma"].map(remove_non_numeric)

obtaining frequencies of forms in LASLA

In [5]:
lasla_tokens_UDset["token-lemma-UD"] = lasla_tokens_UDset["tokenLabel"] + lasla_tokens_UDset["lila_id_lemma_float"].astype(str) + lasla_tokens_UDset["UD_features"]

In [6]:
frequencies = lasla_tokens_UDset["token-lemma-UD"].value_counts()

In [7]:
frequencies_dict = {'frequency': frequencies}

In [8]:
frequencies_df = pd.DataFrame(frequencies_dict)

In [9]:
lasla_forms_freq = lasla_tokens_UDset.drop_duplicates(subset=["token-lemma-UD"]).drop(columns=["token"]).set_index("token-lemma-UD")

In [10]:
for i in tqdm(lasla_forms_freq.index):
    lasla_forms_freq.loc[i,"frequency"] = frequencies_df.loc[i,"frequency"]

100%|█████████████████████████████████████████████████████████████████████████| 72493/72493 [00:03<00:00, 18130.43it/s]


In [11]:
lasla_forms_freq = lasla_forms_freq.sort_values(by=["frequency"], ascending=False).reset_index()

obtaing variant group for each form

# Mapping UD features of LASLA to cells in paralex format of PrinParLat

In [12]:
UD_features = lasla_forms_freq.copy()

In [13]:
UD_features = UD_features.drop_duplicates(subset=["UD_features"]).reset_index()

In [14]:
for col in UD_features:
    if col not in [ "UD_features" , "tokenLabel" , "cell"] :
        UD_features = UD_features.drop(columns = col)

In [15]:
for i in UD_features.index:
    #print(i)
    featureValue_list = UD_features.loc[i,"UD_features"].split("; ")
    #print(featureValue_list)
    for featureValue in featureValue_list:
        feature = re.sub("(https://universaldependencies.org/la/feat/|#.+)","",featureValue)
        feature = re.sub("#.+","",feature)
        value = re.sub("^.+#","",featureValue)
        #print(feature,value)
        UD_features.loc[i,feature] = value

C:\Users\matteo.pellegrini\AppData\Local\Temp\ipykernel_6772\1022588719.py:10: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'Imp' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  UD_features.loc[i,feature] = value
C:\Users\matteo.pellegrini\AppData\Local\Temp\ipykernel_6772\1022588719.py:10: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'LatAnom' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  UD_features.loc[i,feature] = value
C:\Users\matteo.pellegrini\AppData\Local\Temp\ipykernel_6772\1022588719.py:10: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'Ind' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  UD_features.loc[i,feature] = value
C:\Users\mat

In [16]:
UD_features = UD_features.drop(columns=["Degree","InflClass","InflClass%5Bnominal%5D"])

In [17]:
col_list=list()
for col in UD_features:
    if col != "UD_features":
        col_list.append(col)

#UD_features.drop_duplicates(subset = col_list)

In [18]:
UD_features["Tense-Aspect"] = UD_features["Tense"].astype(str) + UD_features["Aspect"].astype(str)

In [19]:
UD_features["cell_mapped"] = ""

mapping single UD features to paralex format wherever possible

In [20]:
mapping = {
    "FutImp" : "fut.",
    "FutPerf" : "fprf.",
    "nanImp" : "nanImp.",
    "nannan" : "nannan.",
    "nanPerf" : "prf.",
    "nanProsp" : "nanProsp.",
    "PastImp" : "iprf.",
    "PastPerf" : "prf.",
    "PqpPerf" : "pprf.",
    "PresImp" : "prs.",
    "Act": "act.",
    "Pass": "pass.",
    "Fin": "",
    "Gdv": "gdv.",
    "Ger": "ger.",
    "Inf": "inf.",
    "Part": "ptcp.",
    "Sup": "sup.",
    "Ind": "ind.",
    "Sub": "sbjv.",
    "Imp": "imp.",
    "Nom": "nom.",
    "Gen": "gen.",
    "Dat": "dat.",
    "Acc": "acc.",
    "Voc": "voc.",
    "Abl": "abl.",
    "Masc": "m.",
    "Fem": "f.",
    "Neut": "n.",
    "Fem,Masc": "f/m.",
    "Fem,Masc,Neut": "f/m/n.",
    "Masc,Neut": "m/n.",
    "1": "1.",
    "2": "2.",
    "3": "3.",
    "Sing": "sg",
    "Plur": "pl"
}

for i in UD_features.index:
    for feature in ["Tense-Aspect" , "Voice" , "VerbForm", "Mood", "Case" , "Gender", "Person" , "Number"]:
        value = UD_features.loc[i,feature]
        if value in mapping:
            paralex = mapping[value]
            UD_features.loc[i,"cell_mapped"] = UD_features.loc[i,"cell_mapped"] + paralex

Some automatic post-hoc refining

In [21]:
for i in UD_features.index:
    cell_mapped = UD_features.loc[i,"cell_mapped"]
    if "nanImp." in cell_mapped:
        #print(cell_mapped,UD_features.loc[i,"tokenLabel"])
        if UD_features.loc[i,"VerbForm"] == "Inf":
            if type(UD_features.loc[i,"Case"]) is float:
                cell_mapped = re.sub("nanImp\.","prs.",cell_mapped)
            else:
                cell_mapped = re.sub("nanImp\.","fut.",cell_mapped)
                cell_mapped = re.sub("inf.","ptcp.",cell_mapped)
        else:
            cell_mapped = re.sub("nanImp\.","prs.",cell_mapped)
        print(cell_mapped)
    if "nannan." in cell_mapped:
        #print(cell_mapped,UD_features.loc[i,"tokenLabel"])
        cell_mapped = re.sub("nannan.","",cell_mapped)
        #print(cell_mapped)
    if "nanProsp" in cell_mapped:
        #print(cell_mapped,UD_features.loc[i,"tokenLabel"])
        #print(UD_features.loc[i,"VerbForm"],type(UD_features.loc[i,"Case"]))
        if UD_features.loc[i,"VerbForm"] == "Ger":
            #print(cell_mapped,UD_features.loc[i,"tokenLabel"])
            cell_mapped = re.sub("nanProsp\.pass\.","",cell_mapped)
            cell_mapped = re.sub("\.(n\.)?sg","",cell_mapped)
            #print(cell_mapped)
        elif UD_features.loc[i,"VerbForm"] == "Gdv":
            #print(cell_mapped,UD_features.loc[i,"tokenLabel"])
            cell_mapped = re.sub("nanProsp\.pass\.","",cell_mapped)
            #print(cell_mapped)
        elif UD_features.loc[i,"VerbForm"] == "Sup":
            #print(cell_mapped,UD_features.loc[i,"tokenLabel"])
            cell_mapped = re.sub("nanProsp\.","",cell_mapped)
            #print(cell_mapped)
        elif UD_features.loc[i,"VerbForm"] == "Part":
                if re.match(".+ur(us|a|um|i|ae|o|a|am|e|orum|arum|is|os|as)$", UD_features.loc[i,"tokenLabel"]):
                    #print(cell_mapped,UD_features.loc[i,"tokenLabel"])
                    cell_mapped = re.sub("nanProsp\.","fut.",cell_mapped)
                    #print(cell_mapped)
                else:
                    #print(cell_mapped,UD_features.loc[i,"tokenLabel"])
                    cell_mapped = cell_mapped + "???"
                    #print(cell_mapped)
    else:
            print(cell_mapped,UD_features.loc[i,"tokenLabel"])
    UD_features.loc[i,"cell_mapped"] = re.sub("\.$","",cell_mapped)

prs.ind.3.sg est
prs.inf.
prs.inf. esse
prs.ind.3.pl sunt
prs.sbjv.3.sg sit
iprf.ind.3.sg erat
prf.ind.3.sg fuit
iprf.sbjv.3.sg esset
prs.act.ind.3.sg potest
fut.ind.3.sg erit
prs.act.inf.
prs.act.inf. posse
prs.act.ind.3.sg habet
prs.act.inf.
prs.act.inf. facere
prs.act.inf.
prs.act.inf. dicere
prf.inf. fuisse
iprf.ind.3.pl erant
prs.sbjv.3.pl sint
prs.ind.1.sg sum
prs.act.sbjv.3.sg possit
prs.ind.2.sg es
iprf.act.sbjv.3.sg posset
prs.act.inf.
prs.act.inf. habere
prf.act.ind.3.sg dedit
prf.act.ind.3.sg fecit
prf.act.ind.3.sg dixit
prs.act.ind.3.sg facit
prs.pass.ind.3.sg uidetur
iprf.sbjv.3.pl essent
prs.act.inf.
prs.act.inf. dare
prf.act.ind.3.sg potuit
prf.act.ind.3.sg iussit
prs.act.ind.1.sg dico
prs.act.ind.3.pl possunt
prs.act.ind.2.sg uis
prf.sbjv.3.sg fuerit
prf.act.ind.3.sg uenit
prs.act.ind.3.pl habent
prs.act.ind.3.sg uenit
prf.act.ind.1.sg dixi
prs.ind.1.pl sumus
prs.act.ind.1.sg possum
iprf.act.sbjv.3.pl possent
pprf.sbjv.3.sg fuisset
iprf.act.ind.3.sg poterat
prs.act.ind.

<>:7: SyntaxWarning: invalid escape sequence '\.'
<>:9: SyntaxWarning: invalid escape sequence '\.'
<>:12: SyntaxWarning: invalid escape sequence '\.'
<>:23: SyntaxWarning: invalid escape sequence '\.'
<>:24: SyntaxWarning: invalid escape sequence '\.'
<>:28: SyntaxWarning: invalid escape sequence '\.'
<>:32: SyntaxWarning: invalid escape sequence '\.'
<>:37: SyntaxWarning: invalid escape sequence '\.'
<>:45: SyntaxWarning: invalid escape sequence '\.'
<>:7: SyntaxWarning: invalid escape sequence '\.'
<>:9: SyntaxWarning: invalid escape sequence '\.'
<>:12: SyntaxWarning: invalid escape sequence '\.'
<>:23: SyntaxWarning: invalid escape sequence '\.'
<>:24: SyntaxWarning: invalid escape sequence '\.'
<>:28: SyntaxWarning: invalid escape sequence '\.'
<>:32: SyntaxWarning: invalid escape sequence '\.'
<>:37: SyntaxWarning: invalid escape sequence '\.'
<>:45: SyntaxWarning: invalid escape sequence '\.'
C:\Users\matteo.pellegrini\AppData\Local\Temp\ipykernel_6772\1441164272.py:7: SyntaxWa

manually established post-hoc corrections

In [22]:
postHoc_corr = {
"fprf.ind.1.pl" : "fprf.act.ind.1.pl" ,
"fprf.ind.1.sg" : "fprf.act.ind.1.sg" ,
"fprf.ind.2.pl" : "fprf.act.ind.2.pl" ,
"fprf.ind.2.sg" : "fprf.act.ind.2.sg" ,
"fprf.ind.3.pl" : "fprf.act.ind.3.pl" ,
"fprf.ind.3.sg" : "fprf.act.ind.3.sg" ,
"pprf.ind.1.pl" : "pprf.act.ind.1.pl" ,
"pprf.ind.1.sg" : "pprf.act.ind.1.sg" ,
"pprf.ind.2.pl" : "pprf.act.ind.2.pl" ,
"pprf.ind.2.sg" : "pprf.act.ind.2.sg" ,
"pprf.ind.3.pl" : "pprf.act.ind.3.pl" ,
"pprf.ind.3.sg" : "pprf.act.ind.3.sg" ,
"pprf.sbjv.1.pl" : "pprf.act.sbjv.1.pl" ,
"pprf.sbjv.1.sg" : "pprf.act.sbjv.1.sg" ,
"pprf.sbjv.2.sg" : "pprf.act.sbjv.2.sg" ,
"pprf.sbjv.3.pl" : "pprf.act.sbjv.3.pl" ,
"pprf.sbjv.3.sg" : "pprf.act.sbjv.3.sg" ,
"prf.ind.1.pl" : "prf.act.ind.1.pl" ,
"prf.ind.1.sg" : "prf.act.ind.1.sg" ,
"prf.ind.2.pl" : "prf.act.ind.2.pl" ,
"prf.ind.2.sg" : "prf.act.ind.2.sg" ,
"prf.ind.3.pl" : "prf.act.ind.3.pl" ,
"prf.ind.3.sg" : "prf.act.ind.3.sg" ,
"prf.inf" : "prf.act.inf" ,
"prf.sbjv.1.pl" : "prf.act.sbjv.1.pl" ,
"prf.sbjv.1.sg" : "prf.act.sbjv.1.sg" ,
"prf.sbjv.2.pl" : "prf.act.sbjv.2.pl" ,
"prf.sbjv.2.sg" : "prf.act.sbjv.2.sg" ,
"prf.sbjv.3.pl" : "prf.act.sbjv.3.pl" ,
"prf.sbjv.3.sg" : "prf.act.sbjv.3.sg" ,
"fut.ptcp.nom.f.sg" : "fut.act.ptcp.nom.f.sg" ,
"fut.ptcp.nom.n.pl" : "fut.act.ptcp.nom.n.pl" ,
"fut.ptcp.acc.n.pl" : "fut.act.ptcp.acc.n.pl" ,
"fut.ptcp.nom.f.pl" : "fut.act.ptcp.nom.f.pl" ,
"fut.ptcp.gen.f.sg" : "fut.act.ptcp.gen.f.sg" ,
"fut.ptcp.acc.f.sg" : "fut.act.ptcp.acc.f.sg" ,
"fut.ptcp.acc.f.pl" : "fut.act.ptcp.acc.f.pl" ,
"fut.ptcp.voc.m.sg" : "fut.act.ptcp.voc.m.sg" ,
"fut.ptcp.nom.m.pl" : "fut.act.ptcp.nom.m.pl" ,
"fut.ptcp.acc.m.pl" : "fut.act.ptcp.acc.m.pl" ,
"fut.ptcp.nom.n.sg" : "fut.act.ptcp.nom.n.sg" ,
"fut.ptcp.nom.m.sg" : "fut.act.ptcp.nom.m.sg" ,
"fut.imp.2.pl" : "fut.act.imp.2.pl" ,
"fut.imp.2.sg" : "fut.act.imp.2.sg" ,
"fut.imp.3.pl" : "fut.act.imp.3.pl" ,
"fut.imp.3.sg" : "fut.act.imp.3.sg" ,
"fut.ind.1.pl" : "fut.act.ind.1.pl" ,
"fut.ind.1.sg" : "fut.act.ind.1.sg" ,
"fut.ind.2.pl" : "fut.act.ind.2.pl" ,
"fut.ind.2.sg" : "fut.act.ind.2.sg" ,
"fut.ind.3.pl" : "fut.act.ind.3.pl" ,
"fut.ind.3.sg" : "fut.act.ind.3.sg" ,
"iprf.ind.1.pl" : "iprf.act.ind.1.pl" ,
"iprf.ind.1.sg" : "iprf.act.ind.1.sg" ,
"iprf.ind.2.pl" : "iprf.act.ind.2.pl" ,
"iprf.ind.2.sg" : "iprf.act.ind.2.sg" ,
"iprf.ind.3.pl" : "iprf.act.ind.3.pl" ,
"iprf.ind.3.sg" : "iprf.act.ind.3.sg" ,
"iprf.sbjv.1.pl" : "iprf.act.sbjv.1.pl" ,
"iprf.sbjv.1.sg" : "iprf.act.sbjv.1.sg" ,
"iprf.sbjv.2.pl" : "iprf.act.sbjv.2.pl" ,
"iprf.sbjv.2.sg" : "iprf.act.sbjv.2.sg" ,
"iprf.sbjv.3.pl" : "iprf.act.sbjv.3.pl" ,
"iprf.sbjv.3.sg" : "iprf.act.sbjv.3.sg" ,
"prs.imp.2.pl" : "prs.act.imp.2.pl" ,
"prs.imp.2.sg" : "prs.act.imp.2.sg" ,
"prs.imp.3.sg" : "prs.act.imp.3.sg" ,
"prs.ind.1.pl" : "prs.act.ind.1.pl" ,
"prs.ind.1.sg" : "prs.act.ind.1.sg" ,
"prs.ind.2.pl" : "prs.act.ind.2.pl" ,
"prs.ind.2.sg" : "prs.act.ind.2.sg" ,
"prs.ind.3.pl" : "prs.act.ind.3.pl" ,
"prs.ind.3.sg" : "prs.act.ind.3.sg" ,
"prs.inf" : "prs.act.inf" ,
"prs.sbjv.1.pl" : "prs.act.sbjv.1.pl" ,
"prs.sbjv.1.sg" : "prs.act.sbjv.1.sg" ,
"prs.sbjv.2.pl" : "prs.act.sbjv.2.pl" ,
"prs.sbjv.2.sg" : "prs.act.sbjv.2.sg" ,
"prs.sbjv.3.pl" : "prs.act.sbjv.3.pl" ,
"prs.sbjv.3.sg" : "prs.act.sbjv.3.sg" ,
}

for i in UD_features.index:
    if UD_features.loc[i,"cell_mapped"] in postHoc_corr:
        UD_features.loc[i,"cell_mapped"] = postHoc_corr[UD_features.loc[i,"cell_mapped"]]

finding remaining problems

In [23]:
latinflexi_cells = pd.read_csv("../LatInfLexi-cells.csv", index_col=0)

In [24]:
latinflexi_cells

,ud,unimorph,cell_romance_aligned,cell_LatInfLexi_v1,POS,frequency,frequency_Antiquitas,frequency_AetasPatrum,frequency_MediumAeuum,frequency_RecentiorLatinitas
cell_id,,,,,,,,,,
prs.act.ind.1.sg,VerbForm=Fin|Mood=Ind|Tense=Pres|Aspect=Imp|Vo...,V;FIN;ACT;PRS;IPFV;IND;1;SG,PRS-IND~1SG,VERB:Fin+Ind+Pres+-+Act+1+Sing+-+-,verb,739362,91759,249823,369678,28102
prs.act.ind.2.sg,VerbForm=Fin|Mood=Ind|Tense=Pres|Aspect=Imp|Vo...,V;FIN;ACT;PRS;IPFV;IND;2;SG,PRS-IND~2SG,VERB:Fin+Ind+Pres+-+Act+2+Sing+-+-,verb,506527,67863,197762,223077,17825
prs.act.ind.3.sg,VerbForm=Fin|Mood=Ind|Tense=Pres|Aspect=Imp|Vo...,V;FIN;ACT;PRS;IPFV;IND;3;SG,PRS-IND~3SG,VERB:Fin+Ind+Pres+-+Act+3+Sing+-+-,verb,2900100,191845,926928,1699035,82292
prs.act.ind.1.pl,VerbForm=Fin|Mood=Ind|Tense=Pres|Aspect=Imp|Vo...,V;FIN;ACT;PRS;IPFV;IND;1;PL,PRS-IND~1PL,VERB:Fin+Ind+Pres+-+Act+1+Plur+-+-,verb,184046,10467,84917,80393,8269
prs.act.ind.2.pl,VerbForm=Fin|Mood=Ind|Tense=Pres|Aspect=Imp|Vo...,V;FIN;ACT;PRS;IPFV;IND;2;PL,PRS-IND~2PL,VERB:Fin+Ind+Pres+-+Act+2+Plur+-+-,verb,103442,9710,43734,48116,1882
...,...,...,...,...,...,...,...,...,...,...
gen.pl,Case=Gen|Number=Plur,N;GEN;PL,NaN,NOUN:Gen+Plur,noun,518465,56573,189629,250017,22246
dat.pl,Case=Dat|Number=Plur,N;DAT;PL,NaN,NOUN:Dat+Plur,noun,835283,115180,310927,382000,27176
acc.pl,Case=Acc|Number=Plur,N;ACC;PL,NaN,NOUN:Acc+Plur,noun,1565479,216650,557840,734860,56129


In [25]:
cell_ls = [ i for i in latinflexi_cells.index.unique() ]

In [26]:
problem_ls = list()
for i in UD_features["cell_mapped"].unique():
    #print(i)
    if "/" not in i and i not in cell_ls:
        problem_ls.append(i)
problem_ls

['fut.sbjv.3.sg',
 'fut.sbjv.3.pl',
 'fut.sbjv.2.sg',
 'fut.sbjv.1.sg',
 'prf.pass.ptcp.m',
 'prf.pass.sbjv.1.sg',
 'prf.pass.ptcp.acc.f',
 '',
 'nanProsp.act.ptcp.acc.???',
 'fut.act.sbjv.3.sg',
 'prs.ptcp.sg',
 'prf.pass.ind.3.sg',
 'prf.pass.ptcp.f',
 'prs.act.ind.2',
 'prf.pass.ptcp.acc.m',
 'prf.pass.ptcp.acc.n',
 'acc.m.pl',
 'prf.pass.ptcp.nom.sg',
 'sg',
 'prf.act.ptcp.nom.m.sg',
 'nanProsp.act.ptcp.acc.sg???',
 'prf.pass.ind.1.sg',
 'nom.m.pl',
 'prf.pass.inf.acc.f.sg',
 'nom.m.sg',
 'fut.sbjv.1.pl',
 'pprf.pass.sbjv.3.pl',
 'prf.pass.ptcp.n.sg',
 'prf.pass.ptcp.m.sg',
 'prf.pass.ptcp',
 'prs.act.ind.3',
 'prs.ptcp.pl',
 'fprf.pass.ind.2.sg',
 'prf.pass.sbjv.2.sg',
 'prs.act.imp.3.sg',
 'fut.act.ptcp.acc.pl',
 'fut.pass.ptcp.acc.f.sg',
 'act.sg',
 'pprf.pass.sbjv.3.sg',
 'prf.pass.ptcp.m.pl',
 'fut.act.ptcp.acc',
 'fprf.pass.ind.3.sg',
 'fut.act.ptcp.acc.sg',
 'pprf.pass.sbjv.1.sg',
 'pprf.pass.ind.3.sg',
 'fut.act.ptcp.acc.m',
 'prs.act.ind.1',
 'prf.pass.ptcp.nom.pl',
 'prf.

In [27]:
UD_paralex_mapping = UD_features[["UD_features","cell_mapped"]]

In [28]:
UD_paralex_mapping = UD_paralex_mapping.set_index("UD_features")

## Mapping cells of nominal/adjectival forms in LatInfLexi to a notation underspecified for gender (to match the one in LASLA)

In [29]:
mapping_underspecified_cells = pd.read_csv("mapping_underspecified_cells.csv",sep="\t")
mapping_underspecified_cells

,cell_underspecified,cells
0,prf.pass.ptcp.acc.m/n.sg,"prf.pass.ptcp.acc.m.sg,prf.pass.ptcp.acc.n.sg"
1,fut.act.ptcp.acc.m/n.sg,"fut.act.ptcp.acc.m.sg,fut.act.ptcp.acc.n.sg"
2,prf.pass.ptcp.abl.m/n.sg,"prf.pass.ptcp.abl.m.sg,prf.pass.ptcp.abl.n.sg"
3,prf.pass.ptcp.abl.f/m/n.pl,"prf.pass.ptcp.abl.f.pl,prf.pass.ptcp.abl.m.pl,..."
4,fut.act.ptcp.acc.m/n.sg,"fut.act.ptcp.acc.m.sg,fut.act.ptcp.acc.n.sg"
...,...,...
241,prf.pass.ptcp.nom.m/n.sg,"prf.pass.ptcp.nom.m.sg,prf.pass.ptcp.nom.n.sg"
242,fut.act.ptcp.abl.m/n.sg,"fut.act.ptcp.abl.m.sg,fut.act.ptcp.abl.n.sg"
243,fut.pass.ptcp.acc.m/n.sg,"fut.pass.ptcp.acc.m.sg,fut.pass.ptcp.acc.n.sg"
244,prf.pass.ptcp.nom.m/n.pl,"prf.pass.ptcp.nom.m.pl,prf.pass.ptcp.nom.n.pl"


In [30]:
for col in mapping_underspecified_cells:
    if col not in ["cell_underspecified","cells"]:
        mapping_underspecified_cells = mapping_underspecified_cells.drop(columns = col)

In [31]:
mapping_underspecified_cells = mapping_underspecified_cells.drop_duplicates().reset_index(drop=True)

In [32]:
mapping_underspecified_cells

,cell_underspecified,cells
0,prf.pass.ptcp.acc.m/n.sg,"prf.pass.ptcp.acc.m.sg,prf.pass.ptcp.acc.n.sg"
1,fut.act.ptcp.acc.m/n.sg,"fut.act.ptcp.acc.m.sg,fut.act.ptcp.acc.n.sg"
2,prf.pass.ptcp.abl.m/n.sg,"prf.pass.ptcp.abl.m.sg,prf.pass.ptcp.abl.n.sg"
3,prf.pass.ptcp.abl.f/m/n.pl,"prf.pass.ptcp.abl.f.pl,prf.pass.ptcp.abl.m.pl,..."
4,prs.act.ptcp.nom.f/m/n.sg,"prs.act.ptcp.nom.f.sg,prs.act.ptcp.nom.m.sg,pr..."
...,...,...
58,prf.pass.ptcp.dat.f/m/n.sg,"prf.pass.ptcp.dat.f.sg,prf.pass.ptcp.dat.m.sg,..."
59,prs.pass.ptcp.dat.f/m/n.sg,"prs.pass.ptcp.dat.f.sg,prs.pass.ptcp.dat.m.sg,..."
60,prs.act.ptcp.abl.f/m.sg,"prs.act.ptcp.abl.f.sg,prs.act.ptcp.abl.m.sg"
61,fut.ptcp.gen.m/n.sg,"fut.ptcp.gen.m.sg,fut.ptcp.gen.n.sg"


In [33]:
for i in mapping_underspecified_cells.index:
    print("index",i,type(i))
    compatible_cells = mapping_underspecified_cells.loc[i,"cells"].split(",")
    print(len(compatible_cells))
    for n in range(len(compatible_cells)):
        print(compatible_cells[n])
        newIndex = len(mapping_underspecified_cells)
        print("newIndex",newIndex)
        mapping_underspecified_cells.loc[newIndex,"cell"] = compatible_cells[n]
        mapping_underspecified_cells.loc[newIndex,"cell_underspecified"] = mapping_underspecified_cells.loc[i,"cell_underspecified"]

index 0 <class 'int'>
2
prf.pass.ptcp.acc.m.sg
newIndex 63
prf.pass.ptcp.acc.n.sg
newIndex 64
index 1 <class 'int'>
2
fut.act.ptcp.acc.m.sg
newIndex 65
fut.act.ptcp.acc.n.sg
newIndex 66
index 2 <class 'int'>
2
prf.pass.ptcp.abl.m.sg
newIndex 67
prf.pass.ptcp.abl.n.sg
newIndex 68
index 3 <class 'int'>
3
prf.pass.ptcp.abl.f.pl
newIndex 69
prf.pass.ptcp.abl.m.pl
newIndex 70
prf.pass.ptcp.abl.n.pl
newIndex 71
index 4 <class 'int'>
3
prs.act.ptcp.nom.f.sg
newIndex 72
prs.act.ptcp.nom.m.sg
newIndex 73
prs.act.ptcp.nom.n.sg
newIndex 74
index 5 <class 'int'>
3
prf.pass.ptcp.dat.f.pl
newIndex 75
prf.pass.ptcp.dat.m.pl
newIndex 76
prf.pass.ptcp.dat.n.pl
newIndex 77
index 6 <class 'int'>
2
prs.act.ptcp.acc.f.sg
newIndex 78
prs.act.ptcp.acc.m.sg
newIndex 79
index 7 <class 'int'>
3
prs.act.ptcp.abl.f.sg
newIndex 80
prs.act.ptcp.abl.m.sg
newIndex 81
prs.act.ptcp.abl.n.sg
newIndex 82
index 8 <class 'int'>
2
gdv.acc.m.sg
newIndex 83
gdv.acc.n.sg
newIndex 84
index 9 <class 'int'>
3
gdv.abl.f.pl
newInde

C:\Users\matteo.pellegrini\AppData\Local\Temp\ipykernel_6772\1835049281.py:9: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'prf.pass.ptcp.acc.m.sg' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  mapping_underspecified_cells.loc[newIndex,"cell"] = compatible_cells[n]


In [34]:
mapping_underspecified_cells = mapping_underspecified_cells.dropna(subset=["cell"]).drop(columns = "cells").set_index("cell")

In [35]:
mapping_underspecified_cells

,cell_underspecified
cell,
prf.pass.ptcp.acc.m.sg,prf.pass.ptcp.acc.m/n.sg
prf.pass.ptcp.acc.n.sg,prf.pass.ptcp.acc.m/n.sg
fut.act.ptcp.acc.m.sg,fut.act.ptcp.acc.m/n.sg
fut.act.ptcp.acc.n.sg,fut.act.ptcp.acc.m/n.sg
prf.pass.ptcp.abl.m.sg,prf.pass.ptcp.abl.m/n.sg
...,...
prs.act.ptcp.abl.m.sg,prs.act.ptcp.abl.f/m.sg
fut.ptcp.gen.m.sg,fut.ptcp.gen.m/n.sg
fut.ptcp.gen.n.sg,fut.ptcp.gen.m/n.sg


finding cells that map to multiple underspecified cells (that are problematic and in most cases due to errors/inconsistencies in LASLA)

In [36]:
cells = list()
cells_toBeChecked = dict()
for i in mapping_underspecified_cells.index:
    if i not in cells:
        cells.append(i)
    else:
        #print(i,mapping_underspecified_cells.loc[i,"cell_underspecified"])
        amb = list()
        for und in mapping_underspecified_cells.loc[i,"cell_underspecified"]:
            amb.append(und)
        cells_toBeChecked[i] = amb

In [37]:
cells_toBeChecked

{'prs.act.ptcp.acc.f.pl': ['prs.act.ptcp.acc.f/m.pl',
  'prs.act.ptcp.acc.f/m/n.pl'],
 'prs.act.ptcp.acc.m.pl': ['prs.act.ptcp.acc.f/m.pl',
  'prs.act.ptcp.acc.f/m/n.pl'],
 'prf.pass.ptcp.abl.m.sg': ['prf.pass.ptcp.abl.m/n.sg',
  'prf.pass.ptcp.abl.f/m/n.sg'],
 'prf.pass.ptcp.abl.n.sg': ['prf.pass.ptcp.abl.m/n.sg',
  'prf.pass.ptcp.abl.f/m/n.sg'],
 'prs.act.ptcp.f': ['prs.act.ptcp.f/m', 'prs.act.ptcp.f/m/n'],
 'prs.act.ptcp.m': ['prs.act.ptcp.f/m', 'prs.act.ptcp.f/m/n'],
 'prs.act.ptcp.acc.f.sg': ['prs.act.ptcp.acc.f/m.sg',
  'prs.act.ptcp.acc.f/m/n.sg'],
 'prs.act.ptcp.acc.m.sg': ['prs.act.ptcp.acc.f/m.sg',
  'prs.act.ptcp.acc.f/m/n.sg'],
 'prs.act.ptcp.gen.f.sg': ['prs.act.ptcp.gen.f/m/n.sg',
  'prs.act.ptcp.gen.f/m.sg'],
 'prs.act.ptcp.gen.m.sg': ['prs.act.ptcp.gen.f/m/n.sg',
  'prs.act.ptcp.gen.f/m.sg'],
 'prf.pass.ptcp.gen.m.sg': ['prf.pass.ptcp.gen.m/n.sg',
  'prf.pass.ptcp.gen.f/m/n.sg'],
 'prf.pass.ptcp.gen.n.sg': ['prf.pass.ptcp.gen.m/n.sg',
  'prf.pass.ptcp.gen.f/m/n.sg'],
 '

manual disambiguation of problematic cases

In [38]:
manual_disambiguation = {'prs.act.ptcp.acc.f.pl': 'prs.act.ptcp.acc.f/m.pl' ,
 'prs.act.ptcp.acc.m.pl': 'prs.act.ptcp.acc.f/m.pl' ,
 'prf.pass.ptcp.abl.m.sg': 'prf.pass.ptcp.abl.m/n.sg',
 'prf.pass.ptcp.abl.n.sg': 'prf.pass.ptcp.abl.m/n.sg',
 'prs.act.ptcp.f': '',
 'prs.act.ptcp.m': '',
 'prs.act.ptcp.acc.f.sg': 'prs.act.ptcp.acc.f/m.sg',
 'prs.act.ptcp.acc.m.sg': 'prs.act.ptcp.acc.f/m.sg',
 'prs.act.ptcp.gen.f.sg': 'prs.act.ptcp.gen.f/m/n.sg',
 'prs.act.ptcp.gen.m.sg': 'prs.act.ptcp.gen.f/m/n.sg',
 'prf.pass.ptcp.gen.m.sg': 'prf.pass.ptcp.gen.m/n.sg',
 'prf.pass.ptcp.gen.n.sg': 'prf.pass.ptcp.gen.m/n.sg',
 'prs.act.ptcp.nom.f.sg': 'prs.act.ptcp.nom.f/m/n.sg',
 'prs.act.ptcp.nom.m.sg': 'prs.act.ptcp.nom.f/m/n.sg',
 'prf.pass.ptcp.abl.m.pl': 'prf.pass.ptcp.abl.f/m/n.pl',
 'prf.pass.ptcp.abl.n.pl': 'prf.pass.ptcp.abl.f/m/n.pl',
 'prf.pass.ptcp.dat.m.sg': 'prf.pass.ptcp.dat.m/n.sg',
 'prf.pass.ptcp.dat.n.sg': 'prf.pass.ptcp.dat.m/n.sg',
 'prs.act.ptcp.abl.f.sg': 'prs.act.ptcp.abl.f/m/n.sg',
 'prs.act.ptcp.abl.m.sg': 'prs.act.ptcp.abl.f/m/n.sg'}

obtaining mapping dictionary

In [39]:
mapping_dict = dict()

for i in mapping_underspecified_cells.index:
    # manually added list of cells that should not be mapped to underspecified cell
    if i not in [ 'prf.pass.ptcp.nom.m.sg', 'prf.pass.ptcp.nom.n.sg', 'prf.pass.ptcp.nom.m.pl', 'prf.pass.ptcp.nom.n.pl', 'prf.pass.ptcp.acc.m.pl', 'prf.pass.ptcp.acc.n.pl', 'gdv.nom.m.sg', 'gdv.nom.n.sg']:
        if i in cells_toBeChecked:
            mapping_dict[i] = manual_disambiguation[i]
        else:
            mapping_dict[i] = mapping_underspecified_cells.loc[i,"cell_underspecified"]

mapping_dict

{'prf.pass.ptcp.acc.m.sg': 'prf.pass.ptcp.acc.m/n.sg',
 'prf.pass.ptcp.acc.n.sg': 'prf.pass.ptcp.acc.m/n.sg',
 'fut.act.ptcp.acc.m.sg': 'fut.act.ptcp.acc.m/n.sg',
 'fut.act.ptcp.acc.n.sg': 'fut.act.ptcp.acc.m/n.sg',
 'prf.pass.ptcp.abl.m.sg': 'prf.pass.ptcp.abl.m/n.sg',
 'prf.pass.ptcp.abl.n.sg': 'prf.pass.ptcp.abl.m/n.sg',
 'prf.pass.ptcp.abl.f.pl': 'prf.pass.ptcp.abl.f/m/n.pl',
 'prf.pass.ptcp.abl.m.pl': 'prf.pass.ptcp.abl.f/m/n.pl',
 'prf.pass.ptcp.abl.n.pl': 'prf.pass.ptcp.abl.f/m/n.pl',
 'prs.act.ptcp.nom.f.sg': 'prs.act.ptcp.nom.f/m/n.sg',
 'prs.act.ptcp.nom.m.sg': 'prs.act.ptcp.nom.f/m/n.sg',
 'prs.act.ptcp.nom.n.sg': 'prs.act.ptcp.nom.f/m/n.sg',
 'prf.pass.ptcp.dat.f.pl': 'prf.pass.ptcp.dat.f/m/n.pl',
 'prf.pass.ptcp.dat.m.pl': 'prf.pass.ptcp.dat.f/m/n.pl',
 'prf.pass.ptcp.dat.n.pl': 'prf.pass.ptcp.dat.f/m/n.pl',
 'prs.act.ptcp.acc.f.sg': 'prs.act.ptcp.acc.f/m.sg',
 'prs.act.ptcp.acc.m.sg': 'prs.act.ptcp.acc.f/m.sg',
 'prs.act.ptcp.abl.f.sg': 'prs.act.ptcp.abl.f/m/n.sg',
 'prs.

In [40]:
def mapping_underspecified_cells_funct(cell):
    
    mapping = mapping_dict

    if cell in mapping:
        cell_underspecified = mapping[cell]
    else:
        cell_underspecified = cell
    
    return cell_underspecified

In [41]:
latinflexi_cells["cell_underspecified"] = latinflexi_cells.index.map(mapping_underspecified_cells_funct)

In [42]:
UD_paralex_mapping["cell_mapped_underspecified"] = UD_paralex_mapping["cell_mapped"].map(mapping_underspecified_cells_funct)

# Obtaining frequencies of cells
Using the mapping and the available information

In [43]:
for i in tqdm(lasla_tokens_UDset.index):
    UD_features = lasla_tokens_UDset.loc[i,"UD_features"]
    if UD_features in UD_paralex_mapping.index.unique():
        lasla_tokens_UDset.loc[i,"cell"] = UD_paralex_mapping.loc[lasla_tokens_UDset.loc[i,"UD_features"],"cell_mapped"]
        #lasla_tokens_UDset.loc[i,"cell_underspecified"] = UD_paralex_mapping.loc[lasla_tokens_UDset.loc[i,"UD_features"],"cell_mapped_underspecified"]
    else:
        print("PROBLEM",UD_features)

  0%|                                                                                       | 0/404654 [00:00<?, ?it/s]C:\Users\matteo.pellegrini\AppData\Local\Temp\ipykernel_6772\1153175885.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'prs.act.imp.2.sg' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  lasla_tokens_UDset.loc[i,"cell"] = UD_paralex_mapping.loc[lasla_tokens_UDset.loc[i,"UD_features"],"cell_mapped"]
100%|███████████████████████████████████████████████████████████████████████| 404654/404654 [00:34<00:00, 11836.06it/s]


In [44]:
lasla_cells_freq = lasla_tokens_UDset["cell"].value_counts().to_frame()

In [45]:
lasla_cells_freq

,count
cell,
prs.act.ind.3.sg,53771
prs.act.inf,40204
prf.act.ind.3.sg,24702
prs.act.ind.3.pl,17174
prs.act.sbjv.3.sg,13919
...,...
pprf.pass.sbjv.3.sg,1
fut.pass.ptcp.acc.m/n.sg,1
prf.act.ptcp.nom.f.sg,1


In [46]:
mapping_underspecified_cells_bySpecifiedCell = mapping_underspecified_cells.reset_index().set_index("cell")
mapping_underspecified_cells_bySpecifiedCell

,cell_underspecified
cell,
prf.pass.ptcp.acc.m.sg,prf.pass.ptcp.acc.m/n.sg
prf.pass.ptcp.acc.n.sg,prf.pass.ptcp.acc.m/n.sg
fut.act.ptcp.acc.m.sg,fut.act.ptcp.acc.m/n.sg
fut.act.ptcp.acc.n.sg,fut.act.ptcp.acc.m/n.sg
prf.pass.ptcp.abl.m.sg,prf.pass.ptcp.abl.m/n.sg
...,...
prs.act.ptcp.abl.m.sg,prs.act.ptcp.abl.f/m.sg
fut.ptcp.gen.m.sg,fut.ptcp.gen.m/n.sg
fut.ptcp.gen.n.sg,fut.ptcp.gen.m/n.sg


In [47]:
for i in latinflexi_cells.index:
    if i in lasla_cells_freq.index:
        latinflexi_cells.loc[i,"frequency_lasla"] = lasla_cells_freq.loc[i,"count"]
    else:
        if i in mapping_dict:
            underspecified_cell = mapping_dict[i]
            nForms = underspecified_cell.count("/")+1
            latinflexi_cells.loc[i,"frequency_lasla"] = lasla_cells_freq.loc[underspecified_cell,"count"] / nForms

In [48]:
latinflexi_cells.to_csv("../LatInfLexi-cells.csv")